<img src="../assets/agent_patterns_04_overview.png" alt="Advanced A2A Handoff 흐름" style="display:block;width:100%;max-width:1200px;margin:0 auto 24px;border-radius:12px;">


![Advanced LLM-Orchestrated A2A Workflow](../assets/workflow_diagrams/agent_to_agent_advanced.svg)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  A2A 서버 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">A2A Protocol 1.0 JSON-RPC Task</h2>
<p style="margin: 7px 0;">먼저 별도 터미널에서 <code>python 4_agent_patterns/main.py fnb-a2a-advanced-local</code>을 실행하세요. Advanced 전용 Agent Card와 A2A JSON-RPC Task를 준비합니다. 이 서버는 03 실습의 Agent 서버와 다른 포트를 사용하므로 동시에 실행할 수 있습니다.</p>
  </div>
</div>


In [4]:
from pathlib import Path
import importlib
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '4_agent_patterns' / 'src').is_dir())
if str(project_root / '4_agent_patterns') not in sys.path:
    sys.path.insert(0, str(project_root / '4_agent_patterns'))
import src.agent_to_agent_advanced.catalog as a2a_catalog_module
import src.agent_to_agent_advanced.orchestration as a2a_module
import src.agent_to_agent_advanced.ui as a2a_ui_module
a2a_catalog_module = importlib.reload(a2a_catalog_module)
a2a_module = importlib.reload(a2a_module)
a2a_ui_module = importlib.reload(a2a_ui_module)
DEFAULT_REQUEST = a2a_catalog_module.DEFAULT_REQUEST
SCENARIO_REQUESTS = a2a_catalog_module.SCENARIO_REQUESTS
list_agents = a2a_module.list_agents
run_handoff = a2a_module.run_handoff
if 'a2a_ui' in globals():
    a2a_ui.close()
a2a_ui = a2a_ui_module.launch_a2a_ui()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>네 개의 <code>/.well-known/agent-card.json</code> Card에서 조직 경계와 Skill·접속 URL을 확인한다.</li><li>구매팀, 제조기획팀 Assistant, IT팀 Assistant가 선택될 때의 TextPart·JSON DataPart와 Task ID를 확인한다.</li><li>제조기획팀 Assistant의 생산기획·OPEX 예산·Cost Center·예산 이관 신청 검토와 IT팀 Assistant의 ERP 데이터·변경관리 검토 artifact를 확인한다.</li><li>사람 승인과 외부 시스템 권한 없이는 예산 이관이나 ERP 반영을 수행하지 않는 경계를 확인한다.</li></ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
    <h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">언제 A2A를 사용할까?</h2>
    <p style="margin: 7px 0;">고정된 내부 Agent를 한 팀이 관리한다면 일반 HTTPS delegation만으로 충분합니다. URL, 요청·응답 JSON, 인증, 재시도, 작업 상태를 팀의 API 계약으로 일관되게 관리하면 A2A는 필수가 아닙니다.</p>
    <table style="width:100%; margin:10px 0 4px; border-collapse:collapse; font-size:.92em;"><thead><tr><th style="text-align:left; padding:7px; border-bottom:1px solid #c7d2e1;">필요</th><th style="text-align:left; padding:7px; border-bottom:1px solid #c7d2e1;">A2A 기능</th></tr></thead><tbody><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">무슨 Agent인지·무엇을 하는지 발견</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Agent Card</b>의 Skill, 입출력 모드, 접속 URL</td></tr><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">장기 작업의 상태·실패·취소 관리</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Task</b> ID와 working / completed / failed 등 표준 lifecycle</td></tr><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">다음 Agent에 구조화 결과 전달</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Message·Artifact·DataPart</b>의 JSON·파일 전달 계약</td></tr><tr><td style="padding:7px;">서로 다른 팀·프레임워크 Agent 연결</td><td style="padding:7px;"><b>JSON-RPC/HTTP</b> 공통 계약으로 내부 구현을 숨긴 상호운용</td></tr></tbody></table>
    <p style="margin: 9px 0 0; color:#4b5f78;"><b>판단:</b> 내부의 고정된 3개 Agent라면 HTTPS handoff가 더 단순할 수 있습니다. 외부·타팀 Agent, 교체 가능한 Agent, 여러 프레임워크 연결이 필요할 때 A2A의 표준화 비용 절감 효과가 커집니다.</p>
  </div>
</div>


In [15]:
import json
from src.agent_to_agent_advanced.catalog import AGENT_URLS, DEFAULT_REQUEST, SCENARIO_REQUESTS
from src.agent_to_agent_advanced.codex_app_server import CODEX_SQL_GENERATOR_AGENT_ID
from src.agent_to_agent_advanced.harness import (
    active_runtime_overlays_for_request, analyse_trace, apply_human_selection,
    load_revision_library, rollback_active_revision,
)
from src.agent_to_agent_advanced.orchestration import list_agents, run_handoff_stream
from src.agent_to_agent_advanced.transport import discover_cards, run_sync, send_task


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. A2A Team Card와 내부 sub-agent 보기</h2><p style="margin: 7px 0;">A2A는 조직 경계만 공개합니다. 설비팀 안에는 현장·엔지니어·관리자 Agent, 구매팀 안에는 구매·원가·재고·MDM Agent가 있습니다. 제조기획팀 Assistant는 생산기획·OPEX 예산·Cost Center 관리·예산 이관 신청 Agent를, IT팀 Assistant는 ERP 데이터·IT 변경관리 Agent를 조율합니다.</p></div></div>


In [16]:
for agent in list_agents():
    print(f"## {agent['agent_label']} ({agent['agent_id']})")
    print(f"설명: {agent['description']}")
    print(f"Skill: {agent['skill_id']} | URL: {agent['url']}")
    print(f"내부 sub-agent: {' → '.join(agent['sub_agents'])}")
    print(f"실행 경계: {agent['execution_boundary']}\n")


## F&B 설비팀 Assistant (fb-facilities-team)
설명: 설비 식별자·현장 관찰·기술 제약·운영 영향을 구조화합니다. 조달·재무·데이터 변경의 판단은 해당 책임 Team의 검토 범위입니다.
Skill: facility_incident_and_recovery | URL: http://127.0.0.1:9211
내부 sub-agent: 현장 운영 Agent → 설비기술 Agent
실행 경계: 모든 업무 판단·delegation·handoff와 필요 시 SAP BTP Grounding Tool 호출은 현장 요청과 누적 artifact를 바탕으로 AI Core LLM이 생성합니다. 코드는 Tool 실행, A2A 전송, 등록 Agent 검증, 반복 한도와 외부 쓰기 차단만 담당합니다.

## F&B 구매팀 Assistant (fb-procurement-team)
설명: 조달·재고·수급 관점의 선택지를 검토합니다. 역할별 읽기 전용 Tool 결과가 있으면 그 범위 안에서만 근거로 사용합니다.
Skill: procurement_and_supply_review | URL: http://127.0.0.1:9212
내부 sub-agent: 구매조달 Agent → 재고·수급 Agent
실행 경계: 모든 업무 판단·delegation·handoff와 필요 시 SAP BTP Grounding Tool 호출은 현장 요청과 누적 artifact를 바탕으로 AI Core LLM이 생성합니다. 코드는 Tool 실행, A2A 전송, 등록 Agent 검증, 반복 한도와 외부 쓰기 차단만 담당합니다.

## 제조기획팀 Assistant (manufacturing-planning-team)
설명: 생산계획, 자재소요, 신규 원부자재와 생산 투입 준비도를 검토합니다. 역할별 읽기 전용 CSV 스냅샷 결과가 있으면 그 범위 안에서만 근거로 사용합니다.
Skill: production_and_material_readiness | URL: http://127.0.0.1:9213


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. UI 대화 흐름을 A2A event로 보기</h2><p style="margin: 7px 0;">UI가 수신하는 Field 요청, Orchestration 판단, Grounding 검색, Team handoff, Artifact, 완료 event를 코드에서 같은 순서로 확인합니다. 외부 ERP 쓰기나 승인 실행은 하지 않습니다.</p></div></div>


In [17]:
scenario_title = 'Codex SQL · 고객 주문 Top 10 조회'
request = dict(SCENARIO_REQUESTS)[scenario_title]  # 직접 요청으로 바꿔도 됩니다.
chat_log, event_log, result = [], [], None
event_labels = {
    'field_handoff_received': '요청 수신', 'orchestration_started': 'Orchestration 진행',
    'grounding_started': 'Grounding 검색', 'grounding_context': 'Grounding 근거',
    'team_handoff_started': 'Team handoff', 'a2a_task_update': 'A2A Task 상태',
    'artifact_received': 'Artifact 수신', 'complete': '완료',
}
for event in run_handoff_stream(request):
    kind = event['type']
    event_log.append(event)
    print(f"[{event_labels.get(kind, kind)}]")
    if kind == 'field_handoff_received':
        chat_log.append({'sender': '요청 부서', 'kind': 'field', 'text': request})
    elif kind == 'grounding_context':
        audit = event.get('audit', {})
        print(f"  문서 {audit.get('document_count', 0)}개 · {audit.get('state', '-')}")
    elif kind == 'team_handoff_started':
        print(f"  대상: {event.get('agent_id', '-')}")
    elif kind == 'artifact_received':
        print(f"  수신 Team: {event['record'].get('producer', '-')}")
    elif kind == 'complete':
        result = event['result']
        print(f"  종료: {result.get('termination', {}).get('state', '-')}")

if result is None:
    raise RuntimeError('A2A 실행 결과를 받지 못했습니다.')


[요청 수신]
[Orchestration 진행]
[orchestration_observation]
[Orchestration 진행]
[Grounding 검색]
[Grounding 근거]
  문서 2개 · grounded
[orchestration_decision]
[middleware]
[Team handoff]
  대상: codex-sql-generator-agent
[A2A Task 상태]
[A2A Task 상태]
[A2A Task 상태]
[Artifact 수신]
  수신 Team: codex-sql-generator-agent
[middleware]
[Orchestration 진행]
[orchestration_observation]
[Orchestration 진행]
[orchestration_decision]
[field_response_delivered]
[완료]
  종료: completed


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 첫 Task · Grounding · Artifact 확인</h2><p style="margin: 7px 0;">UI 카드에 요약된 내용을 원본 실행 결과에서 확인합니다. 첫 Team handoff의 Task plan, Grounding 검색 근거, Team artifact와 middleware만 읽습니다.</p></div></div>


In [18]:
artifacts = result.get('artifacts', [])
first_record = artifacts[0] if artifacts else {}
first_plan = first_record.get('orchestration', {}).get('task_plan', {}) if isinstance(first_record, dict) else {}
print('[첫 Task]')
print('목적:', first_record.get('orchestration', {}).get('purpose', '없음'))
print('계획:', first_plan.get('summary', '없음'))
print('Grounding 기준:', first_plan.get('grounding_basis', '없음'))
for task in first_plan.get('planned_tasks', []):
    print(' -', task)

print('\n[Grounding 검색]')
for search in result.get('grounding', {}).get('searches', []):
    observation = search.get('result', {})
    print(f"- {search.get('query', '-')}: {observation.get('state', '-')} · 문서 {len(observation.get('document_ids', []))}개 · runtime overlay={observation.get('runtime_overlay_applied', False)}")

print('\n[Team Artifact]')
for record in artifacts:
    payload = record.get('artifact', {}).get('payload', {})
    summary = payload.get('team_synthesis', {}).get('summary', '요약 없음')
    print(f"- {record.get('producer', '-')}: {summary}")

print('\n[Middleware]', [item.get('name') for item in result.get('middleware', [])])
print('[종료 상태]', result.get('termination', {}).get('state', '-'))


[첫 Task]
목적: SAP HANA에서 고객별 주문 건수와 순 주문금액 상위 10개를 읽기 전용으로 조회해 주세요.
계획: 고객별 주문 건수와 순 주문금액 상위 10개 결과를 산출할 수 있는지 확인합니다.
Grounding 기준: 공식 조회는 최신 권한과 운영 규정에 따라야 하며 문서만으로 권한을 부여하지 않으므로, 제공된 읽기 전용 조회 역량 범위에서 실제 스키마와 결과를 확인합니다.
 - 고객 주문 데이터에 해당하는 테이블과 컬럼을 확인하고 집계 기준을 식별합니다.
 - 고객별 주문 건수와 순 주문금액을 내림차순으로 집계하는 단일 조회를 작성합니다.
 - 조회 결과에서 상위 10개 고객과 사용된 데이터 범위를 반환합니다.

[Grounding 검색]
- SAP HANA 고객 주문 데이터에서 고객별 주문 건수와 순 주문금액 상위 10개를 조회할 때 적용되는 읽기 전용 조회 원칙, 결과 검증 또는 SQL 작성 요건이 있습니까?: grounded · 문서 2개 · runtime overlay=False

[Team Artifact]
- codex-sql-generator-agent: HANA 조회를 완료했습니다. 12건을 반환했습니다.

[Middleware] ['orchestration_observation', 'grounding_context_collected', 'handoff_envelope_created', 'artifact_received', 'orchestration_observation']
[종료 상태] completed


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #10a37f; background: #edf9f5; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #0b7a5d; letter-spacing: .02em;">🤖 CODEX SQL AGENT · 수동 A2A</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. Codex SQL Generator Agent 직접 호출</h2><p style="margin: 7px 0;">Orchestration의 선택을 기다리지 않고 Codex SQL Agent Card를 직접 발견한 뒤 A2A Task를 전송합니다. Agent는 SQL을 생성할 뿐 실행하지 않으며, 읽기 전용 <code>SELECT</code> 또는 <code>WITH … SELECT</code>만 허용됩니다.</p></div></div>


In [19]:
codex_cards = run_sync(discover_cards({CODEX_SQL_GENERATOR_AGENT_ID: AGENT_URLS[CODEX_SQL_GENERATOR_AGENT_ID]}))
codex_card = codex_cards[CODEX_SQL_GENERATOR_AGENT_ID]
manual_codex_artifact, manual_codex_task = run_sync(send_task(
    codex_card,
    context_id=result['context_id'],
    target_agent_id=CODEX_SQL_GENERATOR_AGENT_ID,
    purpose='고객 주문 Top 10 조회용 읽기 전용 SQL 초안 생성',
    payload={
        'request': request, 'grounding': result.get('grounding', {}),
        'artifacts': artifacts, 'orchestration_observations': result.get('observations', []),
        'human_approvals': [], 'harness_overlay': '',
    },
))
sql_generation = manual_codex_artifact.get('payload', {}).get('sql_generation', {})
print('[A2A Task 상태]', manual_codex_task.get('status', {}).get('state', '-'))
print('\n[생성 SQL]')
print(sql_generation.get('sql') or '생성 불가: ' + ', '.join(sql_generation.get('missing_inputs', [])))
print('\n[설명]', sql_generation.get('explanation', '-'))
print('[안전 경계]', sql_generation.get('safety_notes', '-'))


[A2A Task 상태] TASK_STATE_COMPLETED

[생성 SQL]
WITH customer_totals AS (
    SELECT
        "BUYER_PARTY_ID" AS "CUSTOMER_ID",
        "BUYER_PARTY_NAME" AS "CUSTOMER_NAME",
        "CURRENCY_CODE",
        COUNT(DISTINCT "ORDER_ID") AS "ORDER_COUNT",
        SUM("NET_AMOUNT") AS "TOTAL_NET_AMOUNT"
    FROM "DBADMIN"."CUSTOMER_ORDER"
    GROUP BY
        "BUYER_PARTY_ID",
        "BUYER_PARTY_NAME",
        "CURRENCY_CODE"
),
ranked_customers AS (
    SELECT
        "CUSTOMER_ID",
        "CUSTOMER_NAME",
        "CURRENCY_CODE",
        "ORDER_COUNT",
        "TOTAL_NET_AMOUNT",
        ROW_NUMBER() OVER (
            PARTITION BY "CURRENCY_CODE"
            ORDER BY "TOTAL_NET_AMOUNT" DESC, "ORDER_COUNT" DESC, "CUSTOMER_ID"
        ) AS "CURRENCY_RANK"
    FROM customer_totals
)
SELECT
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "CURRENCY_CODE",
    "ORDER_COUNT",
    "TOTAL_NET_AMOUNT",
    "CURRENCY_RANK"
FROM ranked_customers
WHERE "CURRENCY_RANK" <= 10
ORDER BY "CURRENCY_CODE", "CU

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎 선택 심화</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. Harness Labs 로그 분석</h2><p style="margin: 7px 0;">별도 Probe를 실행하지 않고 방금 수집한 실행 로그와 chat log만 분석합니다. 첫 Task에서 Grounding 검색을 했지만 근거가 없었다면 Grounding 보강안이 우선 후보가 될 수 있습니다.</p></div></div>


In [20]:
analysis = analyse_trace(request, result, chat_log=chat_log)
print('[Harness 결정]', analysis['decision'])
print('[도메인]', analysis['domain'])
print('[판단]', analysis['rationale'])
for option in analysis.get('options', []):
    impact = option.get('impact', {})
    marker = 'AI 추천' if option.get('recommended') else '선택지'
    print(f"\n{option['option_id']}. [{marker}] {option['title']}")
    print('  근거:', '; '.join(option.get('evidence', [])))
    print('  효과:', impact.get('expected_benefit', '-'))
    print('  유지 통제:', impact.get('mandatory_controls', '-'))
    print('  Build 필요:', impact.get('build_required', False))
    for change in option.get('changes', []):
        print(f"  - {change['component']} · {change['target']}: {change['instruction']}")


[Harness 결정] propose
[도메인] hana-sql
[판단] 읽기 전용 조회와 결과 아티팩트는 완료되었지만, 여러 통화가 존재하는 상태에서 환율 기준 없이 통화별 순위를 반환하여 요청한 전체 상위 10개를 확정하지 못했습니다. 실행 전에 통화 비교 기준을 확인하거나, 검증된 환율 규칙이 없으면 명확히 범위를 제한하도록 보완하는 것이 적절합니다.

option_1. [AI 추천] 다중 통화 사전 확인
  근거: 현재 실행은 USD와 EUR 결과를 확인했지만 통화 간 금액을 비교하지 않았습니다.; 환율 기준과 기준일이 없어 전체 상위 10개를 확정하지 못했습니다.
  효과: 통화 비교가 필요한 경우 요청 범위를 실행 전에 확정하고, 통화별 결과와 전체 통합 결과를 혼동하지 않게 합니다.
  유지 통제: 읽기 전용 조회, 실제 스키마 확인, 결과 검증, 감사 추적 및 기존 안전·승인 통제를 유지합니다.
  Build 필요: False
  - agent_prompt · codex-sql-generator-agent: 고객별 금액 순위 요청 시 집계 대상 통화를 먼저 확인한다. 통화가 하나이면 해당 통화의 상위 10개를 산출한다. 통화가 둘 이상이면 공식적으로 검증된 환율 규칙과 기준일이 있는지 확인하고, 없으면 SQL 실행 전에 사용자가 환율 기준·기준일 또는 통화별 순위 중 하나를 선택하도록 한 번만 질문한다. 검증된 환율 규칙 없이 통화 간 금액을 비교하거나 전체 상위 10개라고 표현하지 않는다. 결과에는 통화, 기준일, 데이터 범위, 주문 건수와 순 주문금액을 명시한다.

option_2. [선택지] 통화 비교 결과 표현 가드레일 강화
  근거: 아티팩트는 통화 간 금액을 비교하지 않는다고 설명했지만, 요청자의 원래 목표인 전체 상위 10개는 충족하지 못했습니다.; 종료 사유도 환율 기준 부재로 전체 순위를 확정할 수 없다고 기록합니다.
  효과: 환율 근거가 없는 경우 결과를 통화별 또는 미완료 상태로 명확히 표시하여 결과 해석 오류를 방지합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">6. 사람 선택 · Comment · 런타임 반영</h2><p style="margin: 7px 0;">아래 <code>selected_option</code>과 Comment를 검토한 뒤 실행하세요. 선택값은 revision library에 저장됩니다. 원본 SOP·Prompt·Tool 소스는 변경하지 않습니다.</p></div></div>


In [21]:
if analysis.get('decision') != 'propose':
    raise RuntimeError('채택 가능한 Harness 옵션이 없습니다. 이전 분석 결과를 확인하세요.')
selected_option = next(option['option_id'] for option in analysis['options'] if option.get('recommended'))
reviewer_comment = '선택한 개선안은 기존 승인·읽기 전용 경계를 유지해야 합니다.'  # 필요하면 수정하세요.
revision = apply_human_selection(analysis, selected_option, reviewer_comment)
print('[저장 revision]', revision['revision_id'])
print('[상태]', revision['status'], '| Build:', revision['build_status'])
print('\n[Orchestration overlay]')
print(revision['overlay'])
print('\n[Grounding overlay]')
print(revision['grounding_overlay'] or '없음')
print('\n[Revision Library]', load_revision_library().get('active_by_domain', {}))


[저장 revision] 79845d34-c7f8-45d9-bbdd-89a1beca3c5a
[상태] active | Build: runtime_ready

[Orchestration overlay]
Runtime Harness revision (human-approved):
[Agent Prompt · codex-sql-generator-agent] 고객별 금액 순위 요청 시 집계 대상 통화를 먼저 확인한다. 통화가 하나이면 해당 통화의 상위 10개를 산출한다. 통화가 둘 이상이면 공식적으로 검증된 환율 규칙과 기준일이 있는지 확인하고, 없으면 SQL 실행 전에 사용자가 환율 기준·기준일 또는 통화별 순위 중 하나를 선택하도록 한 번만 질문한다. 검증된 환율 규칙 없이 통화 간 금액을 비교하거나 전체 상위 10개라고 표현하지 않는다. 결과에는 통화, 기준일, 데이터 범위, 주문 건수와 순 주문금액을 명시한다.
[System Prompt · human-review] Human reviewer comment: 선택한 개선안은 기존 승인·읽기 전용 경계를 유지해야 합니다.

[Grounding overlay]
없음

[Revision Library] {'procurement': '77c902e1-8b53-4987-bcf8-8bc2cbfcf08c', 'hana-sql': '79845d34-c7f8-45d9-bbdd-89a1beca3c5a'}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">7. 동일 요청 재실행 · Grounding 반영 · 롤백</h2><p style="margin: 7px 0;">동일 요청을 다시 실행해 runtime overlay가 전달되는지 확인합니다. 결과가 더 좋다고 자동 판정하지 않고 Grounding audit, Team 흐름, 종료 상태만 비교합니다. 마지막 롤백 셀은 실습 종료 후에 실행하세요.</p></div></div>


In [22]:
runtime_harness, runtime_grounding = active_runtime_overlays_for_request(request)
print('[Runtime Harness]', bool(runtime_harness))
print('[Runtime Grounding]', bool(runtime_grounding))
verification_result = None
for event in run_handoff_stream(request, harness_addendum=runtime_harness or None, grounding_overlay=runtime_grounding or None):
    if event['type'] == 'complete':
        verification_result = event['result']
if verification_result is None:
    raise RuntimeError('검증 실행 결과를 받지 못했습니다.')
print('[기준 → 적용 후 종료]', result['termination'].get('state'), '→', verification_result['termination'].get('state'))
print('[기준 → 적용 후 Team]', [item['producer'] for item in result['artifacts']], '→', [item['producer'] for item in verification_result['artifacts']])
for search in verification_result.get('grounding', {}).get('searches', []):
    observation = search.get('result', {})
    print(f"[Grounding] {search.get('query', '-')}: overlay={observation.get('runtime_overlay_applied', False)} · {observation.get('state', '-')}")


[Runtime Harness] True
[Runtime Grounding] False
[기준 → 적용 후 종료] completed → completed
[기준 → 적용 후 Team] ['codex-sql-generator-agent'] → ['codex-sql-generator-agent']
[Grounding] SAP HANA 고객별 주문 건수와 순 주문금액 상위 10개 조회 시 집계 대상 통화가 여러 개인 경우 통화 기준일 환율 또는 통화별 순위 처리 원칙과 읽기 전용 조회 요건: overlay=False · grounded


In [23]:
rolled_back = rollback_active_revision(analysis['domain'])
print(f"롤백 완료: {rolled_back['title']} · {rolled_back['status']}")
print('활성 revision:', load_revision_library().get('active_by_domain', {}).get(analysis['domain'], '없음'))


롤백 완료: 다중 통화 사전 확인 · rolled_back
활성 revision: 없음


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #10a37f; background: #edf9f5; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #0b7a5d; letter-spacing: .02em;">🤖 ADVANCED · CODEX APP SERVER</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. A2A artifact를 Codex 분석 Agent에 전달하기</h2><p style="margin: 7px 0;">앞 단계의 A2A Team artifact를 로컬 <code>codex app-server --stdio</code>에 JSON-RPC로 전달합니다. App Server는 <code>initialize → thread/start → turn/start</code> 순서로 연결하고, <code>agentMessage/delta</code>와 <code>turn/completed</code> notification을 스트리밍합니다. 아래 예제는 <code>read-only</code> sandbox로 실행되며 파일을 변경하지 않습니다.</p><p style="margin: 7px 0 0; color:#4b5f78;">사전 조건: <code>codex</code> CLI 설치 및 로그인. 프로토콜 참고: <a href="https://github.com/openai/codex/blob/main/codex-rs/app-server/README.md" target="_blank">OpenAI Codex App Server README</a></p></div></div>


In [24]:
import json
import shutil
import subprocess
from itertools import count
from queue import Empty, Queue
from threading import Thread
from time import monotonic

class CodexAppServer:
    """Minimal JSON-RPC client for a local `codex app-server --stdio` process."""
    def __init__(self):
        codex = shutil.which('codex')
        if not codex:
            raise RuntimeError('codex CLI를 찾을 수 없습니다. Codex CLI를 설치하고 로그인하세요.')
        self.process = subprocess.Popen(
            [codex, 'app-server', '--stdio'],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True, bufsize=1,
        )
        self._messages, self._events, self._request_ids = Queue(), [], count(1)
        Thread(target=self._read_stdout, daemon=True).start()

    def _read_stdout(self):
        for line in self.process.stdout:
            try:
                self._messages.put(json.loads(line))
            except json.JSONDecodeError:
                pass  # stdout에는 JSON-RPC message만 사용합니다.

    def _send(self, message):
        if self.process.poll() is not None:
            raise RuntimeError('Codex App Server가 종료되었습니다.')
        self.process.stdin.write(json.dumps(message) + '\n')
        self.process.stdin.flush()

    def request(self, method, params=None, timeout=30):
        request_id = next(self._request_ids)
        self._send({'id': request_id, 'method': method, 'params': params or {}})
        deadline = monotonic() + timeout
        while True:
            try:
                message = self._messages.get(timeout=max(0.1, deadline - monotonic()))
            except Empty as error:
                raise TimeoutError(f'{method} 응답을 {timeout}초 안에 받지 못했습니다.') from error
            if message.get('id') != request_id:
                self._events.append(message)
                continue
            if 'error' in message:
                raise RuntimeError(f'{method} 실패: {message["error"]}')
            return message['result']

    def initialize(self):
        result = self.request('initialize', {
            'clientInfo': {'name': 'a2a-advanced-notebook', 'version': '1.0'},
            'capabilities': {},
        })
        self._send({'method': 'initialized'})
        return result

    def next_event(self, timeout=120):
        if self._events:
            return self._events.pop(0)
        try:
            return self._messages.get(timeout=timeout)
        except Empty as error:
            raise TimeoutError(f'App Server event를 {timeout}초 안에 받지 못했습니다.') from error

    def close(self):
        if self.process.poll() is None:
            self.process.terminate()
            try:
                self.process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                self.process.kill()


In [25]:
if 'codex_server' in globals():
    codex_server.close()

codex_server = CodexAppServer()
server_info = codex_server.initialize()
print(f"Connected: {server_info['userAgent']} ({server_info['platformOs']})")


Connected: a2a-advanced-notebook/0.162.1 (Mac OS 26.6.2; arm64) Apple_Terminal/470.2 (a2a-advanced-notebook; 1.0) (macos)


In [26]:
# thread/start: 실제 파일 변경을 막기 위해 ephemeral + read-only + never approval을 사용합니다.
thread = codex_server.request('thread/start', {
    'cwd': str(project_root),
    'ephemeral': True,
    'sandbox': 'read-only',
    'approvalPolicy': 'never',
})['thread']

artifacts = result.get('artifacts', []) if 'result' in globals() else []
artifact_summary = '\n'.join(
    f"- {record['producer']}: {record['artifact']['payload'].get('team_synthesis', {}).get('summary', '분석 완료')}"
    for record in artifacts
) or '- A2A handoff 결과를 먼저 실행한 뒤 다시 실행하세요.'
prompt = f"""
다음은 F&B 운영변경 A2A handoff의 Team artifact입니다.
{artifact_summary}

위 결과에서 사람이 승인해야 할 항목, ERP 반영 전에 확인할 항목, 누락된 근거를 한국어로 간결하게 검토하세요.
분석만 수행하고 파일 변경, 명령 실행, 외부 시스템 호출은 하지 마세요.
""".strip()
turn = codex_server.request('turn/start', {
    'threadId': thread['id'],
    'input': [{'type': 'text', 'text': prompt}],
})['turn']

# notification을 소비합니다. delta는 진행 중인 응답, turn/completed는 terminal event입니다.
while True:
    event = codex_server.next_event()
    method, params = event.get('method'), event.get('params', {})
    if params.get('threadId') != thread['id']:
        continue
    if method == 'agentMessage/delta':
        print(params.get('delta', ''), end='', flush=True)
    elif method == 'item/completed' and params.get('item', {}).get('type') == 'agentMessage':
        print(params['item'].get('text', ''))
    elif method == 'turn/completed' and params.get('turn', {}).get('id') == turn['id']:
        print(f"\n[turn completed] {params['turn'].get('status')}")
        break


조회 결과의 12건 상세값·변경 전후값·적용일이 제공되지 않아 건별 승인 여부는 판단할 수 없습니다. 다만 ERP 반영 전 아래를 확인해야 합니다.

- 사람 승인 필요: 가격/원가, 품목·레시피(BOM), 판매채널, 거래처, 세금코드, 매장 운영시간·메뉴 변경처럼 매출·재고·회계에 영향을 주는 변경.
- ERP 반영 전 확인: 유효 시작·종료일, 대상 매장/법인/플랜트, 품목·단위·통화·세율 매핑, 중복 또는 상충 변경, 이미 전표·발주·재고 이동이 발생한 기간인지.
- 누락 가능 근거: 요청자 및 승인자, 변경 사유, 원본 요청/계약/가격표/메뉴 근거, 영향 분석, 롤백 방안, 테스트 또는 검증 결과, 적용 책임자.

12건별로 변경 유형·기존값·신규값·적용일·근거 문서를 제공하면 승인 대상과 보류 대상을 구분할 수 있습니다.

[turn completed] completed


In [27]:
# 실습이 끝나면 로컬 App Server 프로세스를 종료합니다.
codex_server.close()


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #10a37f; background: #edf9f5; border-radius: 6px; color: #202124;"><div style="font-size: .82em; font-weight: 700; color: #0b7a5d; letter-spacing: .02em;">🤖 ADVANCED · CODEX APP SERVER AGENT</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. Codex SQL Generator Agent 등록하기</h2><p style="margin: 7px 0;">Codex SQL Generator Agent는 <code>http://127.0.0.1:9223</code>의 Agent Card와 A2A Task를 제공합니다. 제공된 SQL dialect·스키마·A2A artifact만 근거로 단일 읽기 전용 <code>SELECT</code> 또는 <code>WITH … SELECT</code> 초안을 생성하며 SQL을 실행하지 않습니다.</p><p style="margin: 7px 0 0; color:#4b5f78;">App Server는 task마다 <code>ephemeral</code>, <code>read-only</code>, <code>approvalPolicy: never</code>로 실행됩니다. INSERT·UPDATE·DELETE·DDL, 파일 변경·명령 실행·외부 시스템 호출은 Agent system prompt와 sandbox에서 금지됩니다. 사전 조건: <code>codex</code> CLI 설치 및 로그인.</p></div></div>


In [28]:
from src.agent_to_agent_advanced.codex_app_server import CODEX_SQL_GENERATOR_AGENT_ID

codex_agent = next(agent for agent in list_agents() if agent['agent_id'] == CODEX_SQL_GENERATOR_AGENT_ID)
print(f"등록됨: {codex_agent['agent_label']} | {codex_agent['url']}")
if 'result' not in globals():
    print('먼저 2번 A2A handoff를 실행하세요.')
else:
    reviews = [record for record in result['artifacts'] if record['producer'] == CODEX_SQL_GENERATOR_AGENT_ID]
    if not reviews:
        print('이번 요청에서는 LLM이 Codex SQL Generator Agent가 필요하지 않다고 판단했습니다.')
    else:
        generated = reviews[-1]['artifact']['payload']['sql_generation']
        print('\n[Codex SQL Generator Agent]')
        print(generated['sql'] or '생성 불가: ' + ', '.join(generated['missing_inputs']))
        print('안전 경계:', generated['safety_notes'])


등록됨: Codex SQL Generator Agent | http://127.0.0.1:9223

[Codex SQL Generator Agent]
WITH customer_totals AS (
    SELECT
        "BUYER_PARTY_ID" AS "CUSTOMER_ID",
        "BUYER_PARTY_NAME" AS "CUSTOMER_NAME",
        "CURRENCY_CODE",
        COUNT(DISTINCT "ORDER_ID") AS "ORDER_COUNT",
        SUM("NET_AMOUNT") AS "TOTAL_NET_AMOUNT"
    FROM "DBADMIN"."CUSTOMER_ORDER"
    GROUP BY
        "BUYER_PARTY_ID",
        "BUYER_PARTY_NAME",
        "CURRENCY_CODE"
), ranked_customers AS (
    SELECT
        "CUSTOMER_ID",
        "CUSTOMER_NAME",
        "CURRENCY_CODE",
        "ORDER_COUNT",
        "TOTAL_NET_AMOUNT",
        ROW_NUMBER() OVER (
            PARTITION BY "CURRENCY_CODE"
            ORDER BY "TOTAL_NET_AMOUNT" DESC, "ORDER_COUNT" DESC, "CUSTOMER_ID"
        ) AS "CURRENCY_RANK"
    FROM customer_totals
)
SELECT
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "CURRENCY_CODE",
    "ORDER_COUNT",
    "TOTAL_NET_AMOUNT",
    "CURRENCY_RANK"
FROM ranked_customers
WHERE "CURRENCY_RA